# Portfolio inspection copy: CW1 Fashion-MNIST

Adapted from the completed coursework notebook. Embedded historical outputs and cloud metadata were removed for privacy and data redistribution control. The original notebook and its outputs are preserved unchanged outside this repository. Verified aggregate results are under `Project1/results`, `Project2/final_evaluation` or `CW2/results`.

Every original code cell is behind `PORTFOLIO_EXECUTE=False`, so Run All in the default state neither imports the ML stack nor trains, evaluates, installs packages, mounts Drive or writes files. Read code and saved result files first. See `../docs/REPRODUCIBILITY.md` before opting in. CW2 combines fitting and test scoring in one historical workflow; it requires all three opt-ins. No execution was used to regenerate the recorded metrics.


In [ ]:
PORTFOLIO_EXECUTE = False
RUN_TRAINING = False
RUN_FINAL_TEST = False
from pathlib import Path
REPO_ROOT = Path.cwd() if (Path.cwd()/"Project1").is_dir() else Path.cwd().parent


# **PROJECT 2: FASHION-MNIST**

This submission copy preserves the executed outputs and locked five-run evidence. Training, reflection saving and final testing are disabled by default; completed results are read from Google Drive.

**Data source:** Fashion-MNIST, Zalando Research. The saved dataset, split and evidence records are authoritative.


## **1. MOUNTING GOOGLE DRIVE**

Connect Google Colab to Google Drive so the notebook can read the saved Fashion-MNIST dataset and keep experiment evidence between sessions. Run this cell and approve the Drive connection.


In [ ]:
if PORTFOLIO_EXECUTE:
    print("Using repository-local files; no Drive mount is required.")
else:
    print("Inspection mode: code not executed.")


## **2. SETTING THE DATASET AND RESULTS FOLDERS**

Define the project paths for the saved Fashion-MNIST dataset, manifest and experiment outputs. The cell verifies that the required input files exist and creates the results directory if needed.


In [ ]:
if PORTFOLIO_EXECUTE:
    from pathlib import Path


    PROJECT_DIR = REPO_ROOT / 'Project2'
    DATA_FILE = PROJECT_DIR / 'data/fashion_mnist_working.npz'
    MANIFEST_FILE = PROJECT_DIR / 'data/dataset_manifest.json'
    RUNS_DIR = PROJECT_DIR / 'my_experiments'
    if not DATA_FILE.is_file() or not MANIFEST_FILE.is_file():
        raise FileNotFoundError(f'Upload the extracted folder first. Expected: {DATA_FILE}')
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print('Dataset:', DATA_FILE)
    print('Your experiment results will be saved to:', RUNS_DIR)
else:
    print("Inspection mode: code not executed.")


## **3. LOADING LIBRARIES AND CHECKING THE GPU**

Import the libraries required for data handling, visualisation, CNN training and evaluation. Set reproducible random seeds, define utility functions and record the current Colab hardware/software environment.


In [ ]:
if PORTFOLIO_EXECUTE:
    import copy, csv, hashlib, io, json, math, platform, random, sys, time
    from datetime import datetime, timezone
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    from torch import nn
    from torch.nn import functional as F
    from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
    from IPython.display import display

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ENVIRONMENT = {'python': sys.version, 'torch': torch.__version__, 'numpy': np.__version__,
                   'platform': platform.platform(), 'device': str(DEVICE),
                   'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    print(ENVIRONMENT)
    if DEVICE.type != 'cuda':
        print('No GPU selected. Sections 2–4 are fine; select a GPU before training.')

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    def save_json(path, value):
        path = Path(path)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
        temporary.replace(path)

    def atomic_torch_save(value, path):
        temporary = path.with_suffix(path.suffix + '.tmp')
        torch.save(value, temporary)
        temporary.replace(path)
else:
    print("Inspection mode: code not executed.")


## **4. LOADING AND VALIDATING FASHION-MNIST**

Load the saved Fashion-MNIST arrays and verify the dataset checksum, dimensions, class counts and fixed train/validation/test split. Confirm that the 10,000 test images remain reserved for final evaluation.


In [ ]:
if PORTFOLIO_EXECUTE:
    manifest = json.loads(MANIFEST_FILE.read_text())
    actual_hash = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
    assert actual_hash == manifest['sha256'], 'Dataset differs from its manifest; do not train.'
    with np.load(DATA_FILE, allow_pickle=False) as data:
        x_train, y_train = data['x_train'], data['y_train']
        x_validation, y_validation = data['x_validation'], data['y_validation']
        train_indices = data['train_source_indices']
        validation_indices = data['validation_source_indices']
    # Test arrays are loaded only in the final evaluation function.
    CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
    assert x_train.shape == (54000, 28, 28) and x_validation.shape == (6000, 28, 28)
    assert x_train.dtype == np.uint8 and x_validation.dtype == np.uint8
    assert len(np.intersect1d(train_indices, validation_indices)) == 0
    assert np.array_equal(np.sort(np.r_[train_indices, validation_indices]), np.arange(60000))
    assert np.array_equal(np.bincount(y_train), np.full(10, 5400))
    assert np.array_equal(np.bincount(y_validation), np.full(10, 600))
    display(pd.DataFrame({'label': range(10), 'category': CLASSES,
                          'training': np.bincount(y_train), 'validation': np.bincount(y_validation)}))
    print('Checks passed. Pixel values:', x_train.min(), 'to', x_train.max())
    print('Dataset source:', manifest['original_source'])
else:
    print("Inspection mode: code not executed.")


**Validation note.** The checksum, shapes, class counts and non-overlapping source indices verify the fixed 54,000/6,000/10,000 split.


## **5. EXAMINING CLOTHING CATEGORIES**

Display three training examples from each Fashion-MNIST class and compare visible features such as sleeves, necklines, garment length and overall silhouette. Use these observations to identify which categories may be difficult to distinguish at 28 × 28 grayscale resolution.


In [ ]:
if PORTFOLIO_EXECUTE:
    fig, axes = plt.subplots(3, 10, figsize=(15, 5))
    for label, category in enumerate(CLASSES):
        examples = np.flatnonzero(y_train == label)[:3]
        for row, index in enumerate(examples):
            axes[row, label].imshow(x_train[index], cmap='gray', vmin=0, vmax=255)
            axes[row, label].axis('off')
            if row == 0:
                axes[row, label].set_title(category, fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Inspection mode: code not executed.")


**Visual note.** Similar low-resolution silhouettes, especially T-shirt/top and Shirt, motivate the later confusion-pair analysis.


## **6. NORMALISING AND AUGMENTING IMAGES**

Scale pixel intensities from 0–255 to 0–1. Apply training-only augmentation using random rotations within ±10° and horizontal flips with probability 0.5, while validation and test images receive only normalisation.


In [ ]:
if PORTFOLIO_EXECUTE:
    def augment(images):
        count = len(images)
        angle = (torch.rand(count, device=images.device) * 20 - 10) * math.pi / 180
        flip = torch.where(torch.rand(count, device=images.device) < .5, -1., 1.)
        theta = torch.zeros(count, 2, 3, device=images.device)
        theta[:, 0, 0] = angle.cos() * flip
        theta[:, 0, 1] = -angle.sin()
        theta[:, 1, 0] = angle.sin() * flip
        theta[:, 1, 1] = angle.cos()
        grid = F.affine_grid(theta, images.size(), align_corners=False)
        return F.grid_sample(images, grid, mode='bilinear', padding_mode='zeros', align_corners=False)

    set_seed(42)
    originals = torch.from_numpy(x_train[:8]).float().unsqueeze(1) / 255
    changed = augment(originals)
    assert changed.shape == originals.shape and torch.isfinite(changed).all()
    fig, axes = plt.subplots(2, 8, figsize=(12, 4))
    for column in range(8):
        for row, images in enumerate([originals, changed]):
            axes[row, column].imshow(images[column, 0], cmap='gray', vmin=0, vmax=1)
            axes[row, column].axis('off')
    axes[0, 0].set_title('Original', fontsize=10)
    axes[1, 0].set_title('Augmented', fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print("Inspection mode: code not executed.")


**Preprocessing note.** Rotation and horizontal flipping apply only to training images; validation and test images receive deterministic scaling.


## **7. DEFINING THE CONVOLUTIONAL NEURAL NETWORK**

Define the compact 2D CNN used in the controlled experiments. The network uses two 3×3 convolution stages with 32 and 64 channels, ReLU, 2×2 MaxPooling, Dropout 0.3 and a 128-unit classifier producing ten logits. Batch Normalisation is enabled only in selected configurations.


In [ ]:
if PORTFOLIO_EXECUTE:
    class FashionCNN(nn.Module):
        def __init__(self, batch_norm=False, channels=(32, 64)):
            super().__init__()
            layers = []
            incoming = 1
            for outgoing in channels:
                layers.append(nn.Conv2d(incoming, outgoing, kernel_size=3, padding=1))
                if batch_norm:
                    layers.append(nn.BatchNorm2d(outgoing))
                layers.extend([nn.ReLU(), nn.MaxPool2d(2)])
                incoming = outgoing
            self.features = nn.Sequential(*layers)
            self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(.3),
                nn.Linear(channels[-1] * 7 * 7, 128), nn.ReLU(), nn.Dropout(.3), nn.Linear(128, 10))

        def forward(self, images):
            return self.classifier(self.features(images))

    model_preview = FashionCNN()
    print(model_preview)
    print('Trainable parameters:', sum(p.numel() for p in model_preview.parameters()))
    assert model_preview(torch.zeros(2, 1, 28, 28)).shape == (2, 10)
else:
    print("Inspection mode: code not executed.")


**Architecture note.** Batch Normalisation is inserted after each convolution only when enabled. The selected model has 421,834 trainable parameters.


## **8. CHOOSING ONE EXPERIMENT**

Select one controlled experiment by choosing its configuration, unique run ID and random seed, then state the hypothesis before training. Start with baseline_20; subsequent runs will test Batch Normalisation, cosine scheduling and their combination under the same 20-epoch budget.


In [ ]:
if PORTFOLIO_EXECUTE:
    EXPERIMENT = 'combined_40'
    RUN_ID = '05_combined_40_seed42'
    SEED = 42
    MY_HYPOTHESIS = 'I expect extending the combined Batch Normalisation and cosine configuration to 40 epochs to allow additional optimisation and potentially improve validation accuracy beyond the 20-epoch combined run.'
    RUN_TRAINING = False  # Submission-safe default; enable only for an intentional launch or genuine resume.

    PLAN = {
        'baseline_20': dict(batch_norm=False, cosine=False, channels=[32,64], epochs=20),
        'batchnorm_20': dict(batch_norm=True, cosine=False, channels=[32,64], epochs=20),
        'cosine_20': dict(batch_norm=False, cosine=True, channels=[32,64], epochs=20),
        'combined_20': dict(batch_norm=True, cosine=True, channels=[32,64], epochs=20),
        'combined_40': dict(batch_norm=True, cosine=True, channels=[32,64], epochs=40),
        'wider_40': dict(batch_norm=True, cosine=True, channels=[64,128], epochs=40),
    }
    CONFIG = dict(**PLAN[EXPERIMENT], experiment=EXPERIMENT, seed=SEED, batch_size=128,
                  learning_rate=.001, dataset_sha256=actual_hash, run_id=RUN_ID,
                  hypothesis=MY_HYPOTHESIS, implementation_version='colab-v1',
                  operator='user-triggered run; AI-assisted scaffold',
                  prior_public_test_observed=True)
    print(CONFIG)
else:
    print("Inspection mode: code not executed.")


**Experiment record.** `RUN_ID` identifies the saved run. `RUN_TRAINING=False` keeps this submission copy in inspection mode; a genuine resume must retain the same run ID and configuration.


## **9. PREPARING TRAINING AND CHECKPOINT RECOVERY**

Define the training, validation and checkpoint-recovery functions. Train with Adam at a learning rate of 0.001 using cross-entropy loss and batches of 128, while saving the checkpoint with the highest validation accuracy and using validation loss to break ties.


In [ ]:
if PORTFOLIO_EXECUTE:
    def as_tensors(images, labels):
        return (torch.from_numpy(images).float().unsqueeze(1).to(DEVICE) / 255,
                torch.from_numpy(labels.astype(np.int64)).to(DEVICE))

    @torch.no_grad()
    def evaluate_arrays(model, images, labels):
        model.eval()
        losses, correct, probabilities = 0., 0, []
        for start in range(0, len(labels), 512):
            target = labels[start:start+512]
            logits = model(images[start:start+512])
            losses += F.cross_entropy(logits, target, reduction='sum').item()
            correct += int((logits.argmax(1) == target).sum().item())
            probabilities.append(logits.softmax(1).cpu().numpy())
        return losses / len(labels), correct / len(labels), np.concatenate(probabilities)

    def run_experiment(config, training_images, training_labels, validation_images, validation_labels):
        if not config['hypothesis'].strip():
            raise ValueError('Write MY_HYPOTHESIS first. Record your reason before seeing the outcome.')
        if config['epochs'] < 20:
            raise ValueError('Coursework experiments must run for at least 20 epochs.')
        run_id = config['run_id']
        if not run_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in run_id):
            raise ValueError('Use only letters, numbers, underscores and hyphens in RUN_ID.')
        if (RUNS_DIR / 'final_selection.json').exists():
            raise RuntimeError('This cycle is frozen for final evaluation. Preserve it before planning any later cycle.')
        folder = RUNS_DIR / run_id
        folder.mkdir(exist_ok=True)
        config_path = folder / 'config.json'
        if config_path.exists():
            if json.loads(config_path.read_text()) != config:
                raise ValueError('This RUN_ID has different settings. Restore them to resume, or choose a new RUN_ID.')
        else:
            save_json(config_path, config)
            save_json(folder / 'environment.json', ENVIRONMENT)
        if (folder / 'summary.json').exists():
            print('Already completed; preserved:', run_id)
            return json.loads((folder / 'summary.json').read_text())
        set_seed(config['seed'])
        model = FashionCNN(config['batch_norm'], config['channels']).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=1e-5) if config['cosine'] else None
        train_images, train_labels = as_tensors(training_images, training_labels)
        val_images, val_labels = as_tensors(validation_images, validation_labels)
        history, best, best_state, best_epoch = [], (-1., -float('inf')), None, None
        checkpoint_path = folder / 'last.pt'
        if checkpoint_path.exists():
            state = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
            if state['config'] != config:
                raise ValueError('Checkpoint configuration mismatch.')
            model.load_state_dict(state['model'])
            optimizer.load_state_dict(state['optimizer'])
            if scheduler: scheduler.load_state_dict(state['scheduler'])
            history, best = state['history'], tuple(state['best'])
            best_state, best_epoch = state['best_state'], state['best_epoch']
            torch.set_rng_state(state['cpu_rng'])
            if DEVICE.type == 'cuda' and state['cuda_rng']:
                torch.cuda.set_rng_state_all(state['cuda_rng'])
            print('Resuming after epoch', len(history))
        for epoch in range(len(history) + 1, config['epochs'] + 1):
            started = time.time()
            model.train()
            permutation = torch.randperm(len(train_labels), device=DEVICE)
            total_loss, correct = 0., 0
            lr = optimizer.param_groups[0]['lr']
            for indices in permutation.split(config['batch_size']):
                batch, target = augment(train_images[indices]), train_labels[indices]
                optimizer.zero_grad(set_to_none=True)
                logits = model(batch)
                loss = F.cross_entropy(logits, target)
                if not torch.isfinite(loss):
                    save_json(folder / 'failure.json', {'epoch': epoch, 'reason': 'Nonfinite training loss'})
                    raise RuntimeError('Nonfinite loss. Last completed epoch and failure record preserved.')
                loss.backward(); optimizer.step()
                total_loss += loss.item() * len(indices)
                correct += int((logits.argmax(1) == target).sum().item())
            val_loss, val_accuracy, _ = evaluate_arrays(model, val_images, val_labels)
            if (val_accuracy, -val_loss) > best:
                best, best_epoch = (val_accuracy, -val_loss), epoch
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            history.append(dict(epoch=epoch, train_loss=total_loss/len(train_labels),
                train_accuracy=correct/len(train_labels), validation_loss=val_loss,
                validation_accuracy=val_accuracy, learning_rate=lr, seconds=time.time()-started))
            if scheduler: scheduler.step()
            state = dict(config=config, model=model.state_dict(), optimizer=optimizer.state_dict(),
                         scheduler=scheduler.state_dict() if scheduler else None, history=history,
                         best=list(best), best_epoch=best_epoch, best_state=best_state,
                         cpu_rng=torch.get_rng_state(),
                         cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else [])
            atomic_torch_save(state, checkpoint_path)
            pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
            print(f'Epoch {epoch}/{config["epochs"]}: train {correct/len(train_labels):.2%}; validation {val_accuracy:.2%}', flush=True)
        atomic_torch_save(best_state, folder / 'best.pt')
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        summary = dict(run_id=run_id, experiment=config['experiment'], epochs=len(history),
                       best_epoch=best_epoch, validation_accuracy=best[0], validation_loss=-best[1],
                       parameters=sum(p.numel() for p in model.parameters()),
                       checkpoint_sha256=hashlib.sha256((folder / 'best.pt').read_bytes()).hexdigest(),
                       completed_utc=datetime.now(timezone.utc).isoformat())
        save_json(folder / 'summary.json', summary)
        return summary
else:
    print("Inspection mode: code not executed.")


## **10. RUNNING THE SELECTED EXPERIMENT**

Run the selected configuration and record epoch-by-epoch training and validation performance. The completed run saves the best validation checkpoint and experiment evidence for later comparison.


In [ ]:
if PORTFOLIO_EXECUTE:
    if RUN_TRAINING:
        if DEVICE.type != 'cuda':
            raise RuntimeError('Select a Colab GPU before launching the experiment.')
        summary = run_experiment(CONFIG, x_train, y_train, x_validation, y_validation)
        display(summary)
    else:
        print('Submission inspection mode: training is disabled; the saved run evidence is retained below.')
else:
    print("Inspection mode: code not executed.")


## **11. PLOTTING LOSS AND ACCURACY**

Load the selected run’s saved history and plot training and validation loss and accuracy across epochs. Save the learning curves to the run folder for later comparison and reporting.


In [ ]:
if PORTFOLIO_EXECUTE:
    current_folder = RUNS_DIR / RUN_ID
    if (current_folder / 'history.csv').exists():
        history = pd.read_csv(current_folder / 'history.csv')
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for axis, metric in zip(axes, ['loss', 'accuracy']):
            axis.plot(history.epoch, history['train_' + metric], label='Training: augmented + dropout')
            axis.plot(history.epoch, history['validation_' + metric], label='Validation: unaltered')
            axis.set(xlabel='Epoch', ylabel=metric.title())
            axis.legend(fontsize=8); axis.grid(alpha=.2)
        fig.suptitle(RUN_ID); plt.tight_layout()
        if RUN_TRAINING:
            fig.savefig(current_folder / 'curves.png', dpi=160)
        else:
            print('Inspection mode: displayed saved history without overwriting curves.png.')
        plt.show()
        display(history.tail())
    else:
        print('No training history for this RUN_ID yet.')
else:
    print("Inspection mode: code not executed.")


**Curve interpretation.** Training uses augmentation and dropout, whereas validation uses unaltered inputs. The selected run reached its best validation accuracy at epoch 36 without a sustained late divergence.


## **12. RECORDING OBSERVATIONS AND DECISIONS**

The completed observation, interpretation and decision are retained below. Saving is disabled in the submission copy so Run All cannot create a new reflection record.


In [ ]:
if PORTFOLIO_EXECUTE:
    MY_OBSERVATION = 'The extended combined run completed 40 epochs and reached a best validation accuracy of 93.82% at epoch 36, improving on the 93.02% result from the 20-epoch combined run. Validation loss also decreased to approximately 0.177.'

    MY_EXPLANATION = 'The extended combined configuration continued improving beyond 20 epochs and produced the strongest validation performance so far. The training and validation curves remained closely aligned in the later epochs, with no clear sustained overfitting. Because the cosine schedule horizon also changed, the improvement cannot be attributed to training duration alone.'

    MY_NEXT_DECISION = 'Retain the 40-epoch combined configuration as the current leading candidate, update the experiment comparison, and decide whether further model widening is justified before final model selection.'
    SAVE_MY_REFLECTION = False

    if SAVE_MY_REFLECTION:
        if not all(v.strip() for v in [MY_OBSERVATION, MY_EXPLANATION, MY_NEXT_DECISION]):
            raise ValueError('Complete all three reflection fields yourself.')
        if not (current_folder / 'summary.json').exists():
            raise RuntimeError('Wait for the experiment to complete before saving its reflection.')
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        save_json(current_folder / ('reflection_' + stamp + '.json'),
                  dict(observation=MY_OBSERVATION, interpretation=MY_EXPLANATION,
                       next_decision=MY_NEXT_DECISION, saved_utc=stamp))
        print('Your reflection saved; earlier versions preserved.')
else:
    print("Inspection mode: code not executed.")


## **13. COMPARING COMPLETED EXPERIMENTS**

Compare all completed experiments using saved validation accuracy, validation loss, best epoch and parameter count. Rank the runs consistently and save the comparison for model-selection evidence.


In [ ]:
if PORTFOLIO_EXECUTE:
    comparison_path = RUNS_DIR / 'comparison.csv'
    packaged_comparison_path = PROJECT_DIR / 'comparison.csv'
    if not comparison_path.exists() and packaged_comparison_path.exists() and not RUN_TRAINING:
        comparison_path = packaged_comparison_path
    rows = [json.loads(p.read_text()) for p in RUNS_DIR.glob('*/summary.json')]
    if comparison_path.exists() and not RUN_TRAINING:
        comparison = pd.read_csv(comparison_path)
        print('Inspection mode: loaded the frozen validation comparison without overwriting it.')
        display(comparison[['run_id','epochs','best_epoch','validation_accuracy','validation_loss','parameters']])
    elif rows:
        comparison = pd.DataFrame(rows).sort_values(['validation_accuracy','validation_loss'], ascending=[False,True])
        display(comparison[['run_id','epochs','best_epoch','validation_accuracy','validation_loss','parameters']])
        if RUN_TRAINING:
            comparison.to_csv(comparison_path, index=False)
    else:
        print('No completed Colab experiments found.')
else:
    print("Inspection mode: code not executed.")


## **14. FROZEN MODEL SELECTION AND FINAL TEST EVIDENCE**

The validation-selected checkpoint and saved final evaluation are inspected by default. `RUN_FINAL_TEST=False` prevents repeated test evaluation or evidence writes.


In [ ]:
if PORTFOLIO_EXECUTE:
    RUN_FINAL_TEST = False
    MY_SELECTION_REASON = (
        'The 40-epoch combined Batch Normalisation and cosine configuration was selected '
        'because it achieved the highest validation accuracy of 93.82%, with a validation '
        'loss of 0.1769. Selection was based on validation performance before this final '
        'Colab test evaluation.'
    )
    def final_test(selection_reason):
        selection_path = RUNS_DIR / 'final_selection.json'
        output = RUNS_DIR / 'final_evaluation'
        if selection_path.exists():
            selection = json.loads(selection_path.read_text())
        else:
            if not selection_reason.strip():
                raise ValueError('Write a selection reason before freezing the cycle.')
            candidates = [json.loads(p.read_text()) for p in RUNS_DIR.glob('*/summary.json')]
            if not candidates: raise ValueError('No completed models to evaluate.')
            winner = sorted(candidates, key=lambda r: (-r['validation_accuracy'], r['validation_loss'], r['run_id']))[0]
            selection = dict(winner=winner, reason=selection_reason, prior_public_test_observed=True,
                             selected_utc=datetime.now(timezone.utc).isoformat())
            save_json(selection_path, selection)
        output.mkdir(exist_ok=True)
        if (output / 'metrics.json').exists():
            return json.loads((output / 'metrics.json').read_text())
        folder = RUNS_DIR / selection['winner']['run_id']
        config = json.loads((folder / 'config.json').read_text())
        assert config['dataset_sha256'] == actual_hash
        assert hashlib.sha256((folder / 'best.pt').read_bytes()).hexdigest() == selection['winner']['checkpoint_sha256']
        model = FashionCNN(config['batch_norm'], config['channels']).to(DEVICE)
        model.load_state_dict(torch.load(folder / 'best.pt', map_location=DEVICE, weights_only=True))
        with np.load(DATA_FILE, allow_pickle=False) as saved:
            test_images, test_labels = saved['x_test'], saved['y_test']
        assert test_images.shape == (10000,28,28)
        assert np.array_equal(np.bincount(test_labels), np.full(10,1000))
        tx,ty = as_tensors(test_images,test_labels)
        loss,accuracy,probabilities = evaluate_arrays(model,tx,ty)
        predictions = probabilities.argmax(1)
        cm = confusion_matrix(test_labels,predictions,labels=range(10))
        assert cm.sum() == 10000 and cm.trace() == int((predictions == test_labels).sum())
        pair = np.triu(cm + cm.T, 1)
        a,b = np.unravel_index(pair.argmax(), pair.shape)
        metrics = dict(test_accuracy=accuracy, test_loss=loss, correct=int(cm.trace()),
                       above_92_percent=accuracy > .92,
                       most_confused_pair=[CLASSES[a],CLASSES[b]], pair_errors=int(pair[a,b]),
                       a_to_b=int(cm[a,b]), b_to_a=int(cm[b,a]),
                       prior_public_test_observed=True,
                       per_class=classification_report(test_labels,predictions,target_names=CLASSES,output_dict=True,zero_division=0))
        np.savez_compressed(output / 'predictions.npz', labels=test_labels,
                            predictions=predictions, probabilities=probabilities)
        pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(output / 'confusion_matrix.csv')
        for matrix,name in [(cm,'counts'),(cm / cm.sum(1,keepdims=True),'normalised')]:
            fig,ax=plt.subplots(figsize=(10,9))
            ConfusionMatrixDisplay(matrix,display_labels=CLASSES).plot(ax=ax,cmap='Blues',xticks_rotation=45,values_format='d' if name=='counts' else '.2f')
            ax.set_title('Final test: ' + name); plt.tight_layout()
            fig.savefig(output / ('confusion_' + name + '.png'),dpi=160); plt.close(fig)
        pair_indices = np.flatnonzero(((test_labels==a)&(predictions==b)) | ((test_labels==b)&(predictions==a)))
        fig,axes=plt.subplots(2,5,figsize=(12,5))
        for ax,index in zip(axes.flat,pair_indices[:10]):
            ax.imshow(test_images[index],cmap='gray')
            ax.set_title(f'#{index}: {CLASSES[test_labels[index]]}\n→ {CLASSES[predictions[index]]}',fontsize=9)
            ax.axis('off')
        for ax in list(axes.flat)[min(10,len(pair_indices)):]: ax.axis('off')
        plt.tight_layout(); fig.savefig(output / 'confused_examples.png',dpi=160); plt.close(fig)
        confidence=probabilities.max(1)
        correct=predictions==test_labels
        choices=[]
        masks=[correct, np.ones(len(correct),dtype=bool),
               (test_labels==a)&(predictions==b), (test_labels==b)&(predictions==a), ~correct]
        reasons=['Highest-confidence correct','Lowest-confidence prediction',
                 'Confusion a to b','Confusion b to a','Highest-confidence error']
        examples=[]
        for position,(mask,reason) in enumerate(zip(masks,reasons)):
            indices=np.flatnonzero(mask)
            indices=indices[np.argsort(confidence[indices] if position==1 else -confidence[indices])]
            for index in indices:
                if int(index) not in choices:
                    choices.append(int(index)); examples.append(dict(test_index=int(index),
                        true_category=CLASSES[test_labels[index]], predicted_category=CLASSES[predictions[index]],
                        confidence=float(confidence[index]), reason=reason)); break
        for index in np.argsort(confidence):
            if len(examples)>=5: break
            if int(index) not in choices:
                choices.append(int(index)); examples.append(dict(test_index=int(index),
                    true_category=CLASSES[test_labels[index]],predicted_category=CLASSES[predictions[index]],
                    confidence=float(confidence[index]),reason='Fallback: low confidence, unused sample'))
        pd.DataFrame(examples).to_csv(output / 'five_inference_examples.csv',index=False)
        save_json(output / 'metrics.json',metrics)
        return metrics

    if RUN_FINAL_TEST:
        metrics = final_test(MY_SELECTION_REASON)
        display(metrics)
    else:
        selection_path = RUNS_DIR / 'final_selection.json'
        metrics_path = RUNS_DIR / 'final_evaluation' / 'metrics.json'
        if not selection_path.exists() and (PROJECT_DIR / 'final_selection.json').exists():
            selection_path = PROJECT_DIR / 'final_selection.json'
        if not metrics_path.exists() and (PROJECT_DIR / 'final_evaluation/metrics.json').exists():
            metrics_path = PROJECT_DIR / 'final_evaluation/metrics.json'
        print('Submission mode: final test is disabled; inspecting the frozen saved record.')
        if selection_path.exists():
            display(json.loads(selection_path.read_text()))
        if metrics_path.exists():
            display(json.loads(metrics_path.read_text()))
else:
    print("Inspection mode: code not executed.")


## **15. COMPLETING THE REPORT EVIDENCE**

Complete the supporting coursework evidence using the saved experiment results, figures and final evaluation outputs. Finalise the experiment log, dataset card and model card, then verify that the report reflects the measured Colab results and documented tuning decisions.
